In [1]:
print("AI Research Paper Assistant - Ready!")

AI Research Paper Assistant - Ready!


In [2]:
!pip install -q sentence-transformers pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.2 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [5]:
pdf_path = "/content/2303.18223v19.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(text))
print("\nFirst 2000 characters:\n")
print(text[:2000])

Number of pages: 144
Characters extracted: 861670

First 2000 characters:

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language mode

In [6]:
chunk_size = 1500
overlap = 200

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 663

First chunk:

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language models (PLMs) have been proposed by pre-
t

In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embedding shape: (663, 384)


In [8]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 663


In [9]:
def search_paper(question, k=5):
    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    results = []

    for i in indices[0]:
        results.append(chunks[i])

    return results

In [10]:
question = "What are the main capabilities of large language models?"

results = search_paper(question)

for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(result[:1000])


--- Result 1 ---

s,
Y. Cao, and K. Narasimhan, “Tree of thoughts: Delib-
erate problem solving with large language models,”
CoRR, vol. abs/2305.10601, 2023.
[530] G. Wang, Y. Xie, Y. Jiang, A. Mandlekar, C. Xiao,
Y. Zhu, L. Fan, and A. Anandkumar, “Voyager: An
open-ended embodied agent with large language
models,”arXiv preprint arXiv:2305.16291, 2023.
[531] X. Jiang, Y. Dong, L. Wang, Q. Shang, and
G. Li, “Self-planning code generation with large
language model,”CoRR, vol. abs/2303.06689, 2023.
[Online]. Available: https://doi.org/10.48550/arXiv.
2303.06689
[532] I. Singh, V . Blukis, A. Mousavian, A. Goyal, D. Xu,
J. Tremblay, D. Fox, J. Thomason, and A. Garg, “Prog-
prompt: Generating situated robot task plans using
large language models,”CoRR, vol. abs/2209.11302,
2022.
[533] B. Liu, Y. Jiang, X. Zhang, Q. Liu, S. Zhang, J. Biswas,
and P . Stone, “LLM+P: empowering large language
models with optimal planning proficiency,”CoRR,
vol. abs/2304.11477, 2023. [Online]. Available:
https:

In [11]:
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embedding shape: (663, 384)


In [12]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(np.array(embeddings).astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 663


In [13]:
question = "What are the main capabilities of large language models?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    5
)

for rank, idx in enumerate(indices[0]):
    print(f"\n--- Result {rank + 1} ---\n")
    print(chunks[idx][:1200])


--- Result 1 ---

s,
Y. Cao, and K. Narasimhan, “Tree of thoughts: Delib-
erate problem solving with large language models,”
CoRR, vol. abs/2305.10601, 2023.
[530] G. Wang, Y. Xie, Y. Jiang, A. Mandlekar, C. Xiao,
Y. Zhu, L. Fan, and A. Anandkumar, “Voyager: An
open-ended embodied agent with large language
models,”arXiv preprint arXiv:2305.16291, 2023.
[531] X. Jiang, Y. Dong, L. Wang, Q. Shang, and
G. Li, “Self-planning code generation with large
language model,”CoRR, vol. abs/2303.06689, 2023.
[Online]. Available: https://doi.org/10.48550/arXiv.
2303.06689
[532] I. Singh, V . Blukis, A. Mousavian, A. Goyal, D. Xu,
J. Tremblay, D. Fox, J. Thomason, and A. Garg, “Prog-
prompt: Generating situated robot task plans using
large language models,”CoRR, vol. abs/2209.11302,
2022.
[533] B. Liu, Y. Jiang, X. Zhang, Q. Liu, S. Zhang, J. Biswas,
and P . Stone, “LLM+P: empowering large language
models with optimal planning proficiency,”CoRR,
vol. abs/2304.11477, 2023. [Online]. Available:
https:

In [14]:
# Find the beginning of the reference section
reference_markers = [
    "\nReferences\n",
    "\nREFERENCES\n",
    "References"
]

reference_position = None

for marker in reference_markers:
    position = text.find(marker)
    if position != -1:
        reference_position = position
        break

if reference_position:
    clean_text = text[:reference_position]
else:
    clean_text = text

print("Original characters:", len(text))
print("Cleaned characters:", len(clean_text))
print("\nEnd of cleaned text:\n")
print(clean_text[-1000:])

Original characters: 861670
Cleaned characters: 602047

End of cleaned text:

 new contents are completed by a number of
volunteers in our team. Here, we add a special part to thank
all the students who have worked very hard on this part
(also including the ones on our author list).
Contribution on Experiments.We would like to sincerely
thank the following people for their hard work involved in
experiments shown in Table 16.
•Xiaoxue Cheng: implement the experiments for evalu-
ation on Language Generation and HaluEval tasks.
•Yuhao Wang: implement the experiments for evalua-
tion on interaction with environment tasks.
•Bowen Zheng: implement the experiments for evalua-
tion on tool manipulation tasks.
Contribution on Tips.We list the following guys for their
contributions on the corresponding numbers of provided
tips for designing prompts in Table 12.
•Xiaolei Wang: T3, O3
•Beichen Zhang: D2, D5
•Zhipeng Chen: D3, D4
•Junjie Zhang: D6
•Bowen Zheng: D7
•Zican Dong: D8
•Xinyu Tang: C2
•Y

In [15]:
chunk_size = 1500
overlap = 200

clean_chunks = []

start = 0

while start < len(clean_text):
    end = start + chunk_size
    clean_chunks.append(clean_text[start:end])
    start += chunk_size - overlap

print("Number of clean chunks:", len(clean_chunks))
print("\nFirst clean chunk:\n")
print(clean_chunks[0])

Number of clean chunks: 464

First clean chunk:

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language models (PLMs) have been propos

In [16]:
clean_embeddings = embedding_model.encode(
    clean_chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", clean_embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding shape: (464, 384)


In [17]:
dimension = clean_embeddings.shape[1]

clean_index = faiss.IndexFlatIP(dimension)

clean_index.add(
    np.array(clean_embeddings).astype("float32")
)

print("Vectors stored:", clean_index.ntotal)

Vectors stored: 464


In [18]:
question = "What are the main capabilities of large language models?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

distances, indices = clean_index.search(
    np.array(question_embedding).astype("float32"),
    5
)

for rank, idx in enumerate(indices[0]):
    print(f"\n--- Result {rank + 1} ---\n")
    print(clean_chunks[idx][:1200])


--- Result 1 ---

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language models (PLMs) have been proposed by pre-
training Transforme

In [19]:
!pip install -q transformers sentencepiece

In [20]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("Language model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [21]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("Language model loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

Language model loaded successfully!


In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded correctly!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 model loaded correctly!


In [23]:
def ask_paper(question, k=3):
    # Step 1: Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # Step 2: Retrieve the most relevant chunks
    distances, indices = clean_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    retrieved_chunks = [clean_chunks[i] for i in indices[0]]

    # Step 3: Combine the retrieved information
    context = "\n\n".join(retrieved_chunks)

    # Step 4: Create a prompt for the language model
    prompt = f"""
Answer the question using only the information provided in the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 5: Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Step 6: Generate the answer
    outputs = llm.generate(
        **inputs,
        max_new_tokens=150
    )

    # Step 7: Convert the generated tokens into text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [24]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: What are the main capabilities of large language models?

Answer:

Scaling on unsupervised pre-training


In [25]:
def ask_paper(question, k=3):
    # Retrieve relevant chunks
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = clean_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    # Keep only the first 700 characters from each retrieved chunk
    retrieved_chunks = [
        clean_chunks[i][:700]
        for i in indices[0]
    ]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Answer the question clearly and briefly using only the context.

Context:
{context}

Question: {question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=120,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [26]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: What are the main capabilities of large language models?

Answer:

model capacity of LLMs.


In [27]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n{'='*60}")
    print(f"RETRIEVED CHUNK {i+1}")
    print(f"{'='*60}\n")
    print(chunk)


RETRIEVED CHUNK 1

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a ma

RETRIEVED CHUNK 2

c idea of language models is intuitive, it is still chal-
lenging to formally explain why LLMs trained by simple
language modeling objectives (e.g.,next token prediction)
can become capable of solving various real-world tasks.
To investigate this problem, a p

In [28]:
# Split the paper into paragraphs
paragraphs = [
    p.strip()
    for p in clean_text.split("\n\n")
    if len(p.strip()) > 100
]

# Create chunks by combining paragraphs
paragraph_chunks = []

current_chunk = ""

for paragraph in paragraphs:
    if len(current_chunk) + len(paragraph) <= 1800:
        current_chunk += paragraph + "\n\n"
    else:
        if current_chunk:
            paragraph_chunks.append(current_chunk.strip())
        current_chunk = paragraph + "\n\n"

if current_chunk:
    paragraph_chunks.append(current_chunk.strip())

print("Number of paragraph chunks:", len(paragraph_chunks))
print("\nFirst paragraph chunk:\n")
print(paragraph_chunks[0][:1800])

Number of paragraph chunks: 1

First paragraph chunk:

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language models (PLMs) have been 

In [29]:
import re

# Clean up excessive whitespace
normalized_text = re.sub(r'\s+', ' ', clean_text).strip()

# Split approximately at sentence boundaries
sentences = re.split(r'(?<=[.!?])\s+', normalized_text)

sentence_chunks = []
current_chunk = ""

for sentence in sentences:
    if len(current_chunk) + len(sentence) <= 1800:
        current_chunk += sentence + " "
    else:
        if current_chunk:
            sentence_chunks.append(current_chunk.strip())
        current_chunk = sentence + " "

if current_chunk:
    sentence_chunks.append(current_chunk.strip())

print("Number of sentence chunks:", len(sentence_chunks))
print("\nFirst sentence chunk:\n")
print(sentence_chunks[0])

Number of sentence chunks: 350

First sentence chunk:

1 A Survey of Large Language Models Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen† Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving from statistical language models to neural language models. Recently, pre-trained language models (PLMs) have been 

In [30]:
sentence_embeddings = embedding_model.encode(
    sentence_chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", sentence_embeddings.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape: (350, 384)


In [31]:
dimension = sentence_embeddings.shape[1]

sentence_index = faiss.IndexFlatIP(dimension)

sentence_index.add(
    np.array(sentence_embeddings).astype("float32")
)

print("Vectors stored:", sentence_index.ntotal)

Vectors stored: 350


In [32]:
question = "What are the main capabilities of large language models?"

question_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

distances, indices = sentence_index.search(
    np.array(question_embedding).astype("float32"),
    3
)

for rank, idx in enumerate(indices[0]):
    print(f"\n{'='*70}")
    print(f"RETRIEVED CHUNK {rank + 1}")
    print(f"{'='*70}\n")
    print(sentence_chunks[idx])


RETRIEVED CHUNK 1

It has used more data 70 for pre-training (i.e.,about 2 trillion tokens), which mainly contributes to the excellent performance. Furthermore, it also conducts a more robust data cleaning process. •Scaling the open-source modes can improve the performance consistently.By comparing the performance of Vicuna (7B) and Vicuna (13B), Pythia (7B) and Pythia (13B), we can see that the models with larger scales mostly perform better than smaller ones on these evaluation tasks, indicating the effectiveness of scaling up the model size. Across different tasks, scaling model is more beneficial for more complex tasks (e.g.,symbolic and mathematical reasoning), where the larger models mostly outperform smaller ones in a large margin. The readers should be note that these findings about open-source language models are limited to the model sizes. We will continually update this part by including the results of larger versions of these models, and also call for the support of comput

In [35]:
def ask_paper(question, k=3):
    # Convert question to embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # Retrieve relevant chunks
    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    retrieved_chunks = [sentence_chunks[i] for i in indices[0]]

    # Use a smaller amount of context
    context_parts = []

    for chunk in retrieved_chunks:
        context_parts.append(chunk[:600])

    context = "\n\n".join(context_parts)

    # Simple instruction for FLAN-T5
    prompt = f"""Based only on the context below, answer the question in 2-3 sentences.

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [34]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: What are the main capabilities of large language models?

Answer:

A Survey of Large Language Models


In [36]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: What are the main capabilities of large language models?

Answer:

task solving


In [37]:
def ask_paper(question, k=3):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    retrieved_chunks = [sentence_chunks[i] for i in indices[0]]

    # Give the model focused context
    context = "\n\n".join(
        [chunk[:500] for chunk in retrieved_chunks]
    )

    prompt = f"""Read the context and answer the question.
Write one complete sentence.

Context:
{context}

Question:
{question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=60,
        min_new_tokens=15,
        num_beams=5,
        no_repeat_ngram_size=2
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [38]:
question = "What are the main capabilities of large language models?"

answer, retrieved_chunks = ask_paper(question)

print("Question:", question)
print("\nAnswer:\n")
print(answer)

Question: What are the main capabilities of large language models?

Answer:

(iv) and (vi) , they can improve the performance consistently. •Scaling the open-source modes


In [39]:
def retrieve_answer(question, k=2):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    results = [sentence_chunks[i] for i in indices[0]]

    return results


question = "What are the main capabilities of large language models?"

results = retrieve_answer(question)

print("Question:", question)
print("\nRelevant information from the paper:\n")

for i, result in enumerate(results, 1):
    print(f"\n--- Retrieved Passage {i} ---\n")
    print(result)

Question: What are the main capabilities of large language models?

Relevant information from the paper:


--- Retrieved Passage 1 ---

It has used more data 70 for pre-training (i.e.,about 2 trillion tokens), which mainly contributes to the excellent performance. Furthermore, it also conducts a more robust data cleaning process. •Scaling the open-source modes can improve the performance consistently.By comparing the performance of Vicuna (7B) and Vicuna (13B), Pythia (7B) and Pythia (13B), we can see that the models with larger scales mostly perform better than smaller ones on these evaluation tasks, indicating the effectiveness of scaling up the model size. Across different tasks, scaling model is more beneficial for more complex tasks (e.g.,symbolic and mathematical reasoning), where the larger models mostly outperform smaller ones in a large margin. The readers should be note that these findings about open-source language models are limited to the model sizes. We will continually u

In [40]:
question = "What special abilities do large language models exhibit?"

results = retrieve_answer(question, k=3)

print("Question:", question)
print("\nRelevant information from the paper:\n")

for i, result in enumerate(results, 1):
    print(f"\n--- Retrieved Passage {i} ---\n")
    print(result[:1200])

Question: What special abilities do large language models exhibit?

Relevant information from the paper:


--- Retrieved Passage 1 ---

1 A Survey of Large Language Models Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen† Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving from statistical language models t

In [41]:
def paper_assistant(question, k=3):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    print("Question:", question)
    print("\nRelevant information from the paper:\n")

    for rank, idx in enumerate(indices[0], 1):
        print(f"\n--- Evidence {rank} ---")
        print(sentence_chunks[idx][:1000])

In [42]:
paper_assistant(
    "What special abilities do large language models exhibit?"
)

Question: What special abilities do large language models exhibit?

Relevant information from the paper:


--- Evidence 1 ---
1 A Survey of Large Language Models Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen† Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving from statistical language models to neural l